# Axis 1 — Cache Capacity vs Working Set (Mac / MLX)

**Model:** `allenai/OLMoE-1B-7B-0125-Instruct` (unquantized F16, MLX format)  
**Architecture:** 16 layers × 64 experts/layer × top-8 routing = 1,024 logical experts  
**Platform:** Apple Silicon, MLX

## True Disk-Backed Expert Cache

This notebook implements a **true K-expert cache** where:

1. Only **non-expert parameters** (~0.5 GB) are loaded into memory via selective `safe_open`
2. Expert weights **stay on disk** in safetensors files
3. On cache miss, **only the requested expert's bytes** are read from disk (~12 MB per expert)
4. At most **K experts per layer** are resident in memory at any time
5. LRU eviction when the cache is full

The complete expert bank (~12.3 GB) is **never** resident in memory.

In [1]:
import gc, time, struct, json, platform
from collections import OrderedDict, Counter
from pathlib import Path

import numpy as np
import pandas as pd
import psutil
import mlx.core as mx
import mlx.nn as nn
from safetensors import safe_open
from mlx_lm.utils import load_config, _get_classes
from mlx_lm.tokenizer_utils import load as load_tokenizer
from mlx_lm.generate import generate_step

assert platform.system() == 'Darwin', 'This notebook requires macOS with Apple Silicon.'

## Configuration

In [2]:
# ── Paths ──
cwd = Path.cwd()
if cwd.name == 'axis-1-cache-capacity':
    MODEL_DIR = cwd.parent.parent / 'models' / 'OLMoE-1B-7B-0125-Instruct'
else:
    MODEL_DIR = cwd / 'models' / 'OLMoE-1B-7B-0125-Instruct'

assert MODEL_DIR.exists(), f'Model not found: {MODEL_DIR}'

# ── Experiment parameters ──
CACHE_CAPACITY_K = 62          # K = max experts resident per layer
MAX_NEW_TOKENS   = 24         # tokens to generate
PROMPT = 'Explain why loading only the experts selected by a router can reduce memory use.'

print(f'Model dir: {MODEL_DIR}')
print(f'Cache capacity: K = {CACHE_CAPACITY_K} per layer')
print(f'Total max resident experts: {CACHE_CAPACITY_K} × 16 layers = {CACHE_CAPACITY_K * 16}')

Model dir: /Users/falak/Documents/Falak/College/Sem - 3/DS635/MoE - Inference System/models/OLMoE-1B-7B-0125-Instruct
Cache capacity: K = 62 per layer
Total max resident experts: 62 × 16 layers = 992


## Step 1: Build Expert Disk Index

Parse safetensors file headers to locate each expert's byte range on disk.  
No tensor data is loaded — only the JSON metadata header (a few KB per shard).

Expert tensors are individual per-expert weights in F16.  
Each expert projection is ~2–4 MB; three projections (gate, up, down) = ~12 MB per expert.

In [3]:
def parse_safetensors_header(filepath):
    """Read only the JSON header from a safetensors file (no tensor data)."""
    with open(filepath, 'rb') as f:
        header_size = struct.unpack('<Q', f.read(8))[0]
        header_json = json.loads(f.read(header_size))
    return header_size, header_json

expert_index = {}                 # (layer, expert_id, proj_name) -> disk location info
non_expert_keys_by_shard = {}     # shard_file -> [non-expert tensor names]
shard_files = sorted(MODEL_DIR.glob('model*.safetensors'))
if not shard_files:
    raise FileNotFoundError(f'No model*.safetensors files found in {MODEL_DIR}. Is the download finished?')

for shard_file in shard_files:
    header_size, metadata = parse_safetensors_header(shard_file)
    data_region_start = 8 + header_size
    ne_keys = []

    for tensor_name, info in metadata.items():
        if tensor_name == '__metadata__':
            continue

        # Expert tensors: model.layers.{L}.mlp.experts.{E}.{proj}.weight
        is_expert = '.mlp.experts.' in tensor_name and tensor_name.endswith('.weight')

        if is_expert:
            parts = tensor_name.split('.')
            layer_idx = int(parts[2])
            expert_id = int(parts[5])
            proj_name = parts[6]           # gate_proj / up_proj / down_proj

            expert_index[(layer_idx, expert_id, proj_name)] = {
                'file': str(shard_file),
                'data_start': data_region_start + info['data_offsets'][0],
                'expert_stride': info['data_offsets'][1] - info['data_offsets'][0],
                'expert_shape': tuple(info['shape']),
                'dtype_str': info['dtype'],
            }
        else:
            ne_keys.append(tensor_name)

    non_expert_keys_by_shard[str(shard_file)] = ne_keys

# ── Summary ──
n_expert_entries = len(expert_index)
n_non_expert = sum(len(v) for v in non_expert_keys_by_shard.values())
sample = expert_index[(0, 0, 'gate_proj')]

# Compute per-expert byte total (3 projections)
expert_bytes_total = sum(
    expert_index[(0, 0, p)]['expert_stride'] for p in ['gate_proj', 'up_proj', 'down_proj']
)

print(f'Expert index: {n_expert_entries} entries (expect 3072 = 16 layers × 64 experts × 3 projections)')
print(f'Non-expert tensors: {n_non_expert}')
print(f'\nSample — layer 0, expert 0, gate_proj:')
print(f'  Shape: {sample["expert_shape"]}')
print(f'  Bytes: {sample["expert_stride"]:,} ({sample["expert_stride"]/1e6:.1f} MB)')
print(f'  Dtype: {sample["dtype_str"]}')
print(f'  Shard: {Path(sample["file"]).name}')
print(f'\nPer-expert total (3 projections): {expert_bytes_total/1e6:.1f} MB')

Expert index: 3072 entries (expect 3072 = 16 layers × 64 experts × 3 projections)
Non-expert tensors: 147

Sample — layer 0, expert 0, gate_proj:
  Shape: (1024, 2048)
  Bytes: 4,194,304 (4.2 MB)
  Dtype: F16
  Shard: model-00001-of-00003.safetensors

Per-expert total (3 projections): 12.6 MB


## Step 2: Verify Disk Byte-Range Read

Before building the cache, verify that reading raw bytes at computed offsets
produces the **exact same** weight tensor as `safe_open` selective loading.

The model stores expert weights in **F16** (not BF16), so `np.float16` is the correct dtype.

In [4]:
# Load expert 0 via safe_open (ground truth)
ie = expert_index[(0, 0, 'gate_proj')]
with safe_open(ie['file'], framework='numpy') as sf:
    ref_e0 = mx.array(sf.get_tensor('model.layers.0.mlp.experts.0.gate_proj.weight'))
mx.eval(ref_e0)

# Load same expert via raw byte-range read
with open(ie['file'], 'rb') as f:
    f.seek(ie['data_start'])
    raw = f.read(ie['expert_stride'])

# On-disk dtype is F16 → read as np.float16
disk_e0 = mx.array(
    np.frombuffer(raw, dtype=np.float16).reshape(ie['expert_shape'])
)
mx.eval(disk_e0)

diff = mx.max(mx.abs(ref_e0 - disk_e0)).item()
print(f'Layer 0, gate_proj, expert 0 — safe_open vs byte-range: max_diff = {diff}')
assert diff == 0.0, f'Byte-range read verification FAILED: max_diff = {diff}'
print(f'✓ Byte-range read produces identical weights (dtype: {ie["dtype_str"]})')

del ref_e0, disk_e0, raw
gc.collect()

Layer 0, gate_proj, expert 0 — safe_open vs byte-range: max_diff = 0.0
✓ Byte-range read produces identical weights (dtype: F16)


16

## Step 3: Load Non-Expert Weights Only

Create the model architecture, then load **only** non-expert parameters:
- Embeddings, layer norms, attention projections, router gate weights
- Approximately 0.5 GB

Uses `safetensors.safe_open` for **selective tensor loading** — only the requested
tensor names are read from disk. Expert weights are never touched.

After loading, the randomly-initialized expert weight tensors from `SwitchGLU.__init__`
are replaced with tiny placeholders and garbage-collected, so they never consume memory.

In [5]:
config = load_config(MODEL_DIR)
tokenizer = load_tokenizer(MODEL_DIR)

model_class, model_args_class = _get_classes(config)
model = model_class(model_args_class.from_dict(config))

rss_before = psutil.Process().memory_info().rss
print('Loading ONLY non-expert weights via selective safe_open...')

for shard_file in shard_files:
    ne_keys = non_expert_keys_by_shard[str(shard_file)]
    if not ne_keys:
        continue
    with safe_open(str(shard_file), framework='numpy') as sf:
        weights = {}
        for key in ne_keys:
            weights[key] = mx.array(sf.get_tensor(key))
    model.load_weights(list(weights.items()), strict=False)
    del weights
    gc.collect()

# Clear randomly-initialized expert weights from SwitchGLU to prevent
# the full expert bank from ever being evaluated/resident.
for i in range(len(model.model.layers)):
    old_switch = model.model.layers[i].mlp.switch_mlp
    for proj in ['gate_proj', 'up_proj', 'down_proj']:
        sl = getattr(old_switch, proj)
        sl.weight = mx.zeros((1, 1, 1))

gc.collect()
mx.clear_cache()

rss_after = psutil.Process().memory_info().rss
print(f'Done. RSS: {rss_before/1e9:.2f} → {rss_after/1e9:.2f} GB '
      f'(+{(rss_after-rss_before)/1e9:.2f} GB)')

# Sanity: router weights are loaded correctly
gate_w = model.model.layers[0].mlp.gate['weight']
mx.eval(gate_w)
print(f'Router gate weight shape: {gate_w.shape} ✓')

Loading ONLY non-expert weights via selective safe_open...
Done. RSS: 0.61 → 1.77 GB (+1.16 GB)
Router gate weight shape: (64, 2048) ✓


## Step 4: Disk-Backed Expert Cache

Each layer gets a `DiskBackedExpertCache` with capacity K.  
On cache miss, it reads a single expert's 3 projections (~12 MB) from disk  
via `seek()` + `read()` at the pre-computed byte offset. No full tensor materialization.

The cache replaces `switch_mlp` on each `OlmoeSparseMoeBlock`, so the router
calls `cache(x_flat, indices)` instead of the original `SwitchGLU`.

Per-expert computation: `out = silu(x @ gate.T) * (x @ up.T) @ down.T`  
(numerically equivalent to `gather_mm`-based `SwitchGLU`).

In [6]:
class DiskBackedExpertCache:
    """Per-layer LRU expert cache.
    
    Expert weights stay on disk in safetensors files.
    Only `capacity` experts are resident in memory at any time.
    On a miss, reads the expert's byte range from disk (~12 MB).
    """

    def __init__(self, layer_idx, expert_index, capacity=4):
        self.layer_idx = layer_idx
        self.capacity = max(1, int(capacity))
        self.cache = OrderedDict()   # expert_id -> (gate_w, up_w, down_w)
        
        # Counters
        self.hits = 0
        self.misses = 0
        self.evictions = 0
        self.bytes_loaded = 0
        self.load_time_ms = 0.0
        self.accesses = Counter()

        # Pre-resolved disk locations for this layer's 3 projections
        self.expert_index = expert_index

    def _read_expert(self, eid, proj_name):
        """Read one expert projection's weight from disk via byte-range seek."""
        ie = self.expert_index[(self.layer_idx, eid, proj_name)]
        with open(ie['file'], 'rb') as f:
            f.seek(ie['data_start'])
            raw = f.read(ie['expert_stride'])
        # On-disk dtype is F16
        arr = np.frombuffer(raw, dtype=np.float16).reshape(ie['expert_shape'])
        return mx.array(arr)

    def get(self, eid):
        """Get expert weights, loading from disk on miss. Returns (gate_w, up_w, down_w)."""
        eid = int(eid)
        self.accesses[eid] += 1

        if eid in self.cache:
            self.hits += 1
            w = self.cache.pop(eid)      # move to end (most recently used)
            self.cache[eid] = w
            return w

        # Cache miss — load from disk
        self.misses += 1
        t0 = time.perf_counter()
        gw = self._read_expert(eid, 'gate_proj')
        uw = self._read_expert(eid, 'up_proj')
        dw = self._read_expert(eid, 'down_proj')
        mx.eval(gw, uw, dw)  # Force evaluation so timing is accurate
        self.load_time_ms += (time.perf_counter() - t0) * 1000
        self.bytes_loaded += sum(
            self.expert_index[(self.layer_idx, eid, p)]['expert_stride']
            for p in ['gate_proj', 'up_proj', 'down_proj']
        )

        w = (gw, uw, dw)
        self.cache[eid] = w

        # Evict LRU if over capacity
        if len(self.cache) > self.capacity:
            _, evicted = self.cache.popitem(last=False)
            del evicted
            self.evictions += 1

        return w

    def __call__(self, x_flat, indices):
        """Drop-in replacement for SwitchGLU.__call__.
        
        Args:
            x_flat: (N, D) input tensor
            indices: (N, top_k) expert indices from router
        Returns:
            (N, top_k, D) output tensor
        """
        N, k = indices.shape
        il = indices.tolist()

        if N == 1:  # ── Decode: single token ──
            y = []
            for eid in il[0]:
                gw, uw, dw = self.get(eid)
                g = x_flat @ gw.T
                u = x_flat @ uw.T
                h = nn.silu(g) * u
                y.append(h @ dw.T)
            return mx.stack(y, axis=1)

        # ── Prefill: batch by unique expert for efficiency ──
        expert_map = {}  # eid -> [(token_idx, slot_idx), ...]
        for t in range(N):
            for p in range(k):
                eid = il[t][p]
                expert_map.setdefault(eid, []).append((t, p))

        y = [[None] * k for _ in range(N)]
        for eid, positions in expert_map.items():
            gw, uw, dw = self.get(eid)
            token_indices = [t for t, p in positions]
            xt = x_flat[token_indices]         # (M, D)
            g = xt @ gw.T                      # (M, intermediate)
            u = xt @ uw.T
            h = nn.silu(g) * u
            out = h @ dw.T                     # (M, D)
            for i, (t, p) in enumerate(positions):
                y[t][p] = out[i]

        output = mx.stack([mx.stack(row) for row in y])
        mx.eval(output)
        return output

    def resident_bytes(self):
        """Calculate actual bytes used by resident expert tensors."""
        total = 0
        for eid, (gw, uw, dw) in self.cache.items():
            total += gw.nbytes + uw.nbytes + dw.nbytes
        return total

    def summary(self):
        """Return a dict of cache statistics for this layer."""
        total = self.hits + self.misses
        return {
            'layer': self.layer_idx,
            'K': self.capacity,
            'resident': len(self.cache),
            'accesses': total,
            'hits': self.hits,
            'misses': self.misses,
            'hit_rate': f'{self.hits/total:.1%}' if total else 'N/A',
            'evictions': self.evictions,
            'MB_loaded': round(self.bytes_loaded / 1e6, 1),
            'io_ms': round(self.load_time_ms, 1),
            'unique_experts': len(self.accesses),
            'resident_MB': round(self.resident_bytes() / 1e6, 1),
        }


## Step 5: Install Cache on All Layers

Replace `switch_mlp` on each `OlmoeSparseMoeBlock` with a `DiskBackedExpertCache`.  
After this cell, the model's memory footprint should be ~0.5–1.0 GB (non-expert params only).

In [7]:
layer_caches = {}
for i in range(len(model.model.layers)):
    cache = DiskBackedExpertCache(i, expert_index, capacity=CACHE_CAPACITY_K)
    layer_caches[i] = cache
    model.model.layers[i].mlp.switch_mlp = cache

gc.collect()
mx.clear_cache()

metal_mem = mx.get_active_memory() / 1e9
rss_mem = psutil.Process().memory_info().rss / 1e9

print(f'Installed DiskBackedExpertCache on {len(layer_caches)} layers (K={CACHE_CAPACITY_K})')
print(f'Active Metal memory: {metal_mem:.2f} GB')
print(f'RSS: {rss_mem:.2f} GB')

Installed DiskBackedExpertCache on 16 layers (K=62)
Active Metal memory: 0.95 GB
RSS: 1.77 GB


## Step 6: Generate with Cache

Run end-to-end generation using `generate_step` (low-level API).  
The cache will load experts from disk on miss during both prefill and decode.

In [8]:
# Tokenize prompt
messages = [{'role': 'user', 'content': PROMPT}]
prompt_str = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=False
)
prompt_tokens = mx.array(tokenizer.encode(prompt_str))

print(f'Prompt: "{PROMPT}"')
print(f'Prompt tokens: {len(prompt_tokens)}')
print(f'Max new tokens: {MAX_NEW_TOKENS}')
print(f'\nGenerating...')

# ── Generation loop ──
t_start = time.perf_counter()
generated_tokens = []

for token, logprobs in generate_step(prompt_tokens, model, max_tokens=MAX_NEW_TOKENS):
    generated_tokens.append(token)
    mx.eval(token)      # Ensure the token is actually computed

t_total = time.perf_counter() - t_start
generated_text = tokenizer.decode(generated_tokens)

print(f'\n=== GENERATION RESULT ===')
print(f'Tokens: {len(generated_tokens)}')
print(f'Wall time: {t_total*1000:.0f} ms')
print(f'Tokens/sec: {len(generated_tokens)/t_total:.2f}')
print(f'Response: {generated_text}')

print(f'\n=== MEMORY ===')
print(f'Active Metal memory: {mx.get_active_memory()/1e9:.2f} GB')
print(f'Peak Metal memory: {mx.get_peak_memory()/1e9:.2f} GB')

Prompt: "Explain why loading only the experts selected by a router can reduce memory use."
Prompt tokens: 29
Max new tokens: 24

Generating...

=== GENERATION RESULT ===
Tokens: 24
Wall time: 19078 ms
Tokens/sec: 1.26
Response: Loading only the experts selected by a router can significantly reduce memory use for several reasons:

1. **Reduced

=== MEMORY ===
Active Metal memory: 12.58 GB
Peak Metal memory: 12.61 GB


## Step 7: Cache Statistics

In [9]:
# ── Per-layer statistics ──
stats = pd.DataFrame([c.summary() for c in layer_caches.values()])
print('Per-layer cache statistics:')
print(stats.to_string(index=False))

# ── Global summary ──
total_accesses  = sum(c.hits + c.misses for c in layer_caches.values())
total_hits      = sum(c.hits for c in layer_caches.values())
total_misses    = sum(c.misses for c in layer_caches.values())
total_evictions = sum(c.evictions for c in layer_caches.values())
total_io_ms     = sum(c.load_time_ms for c in layer_caches.values())
total_bytes     = sum(c.bytes_loaded for c in layer_caches.values())
total_resident  = sum(c.resident_bytes() for c in layer_caches.values())

print(f'\n{"="*50}')
print(f'GLOBAL CACHE SUMMARY  (K = {CACHE_CAPACITY_K} per layer)')
print(f'{"="*50}')
print(f'Total accesses:       {total_accesses}')
print(f'Total hits:           {total_hits} ({total_hits/total_accesses:.1%})')
print(f'Total misses:         {total_misses} ({total_misses/total_accesses:.1%})')
print(f'Total evictions:      {total_evictions}')
print(f'Data loaded from disk:{total_bytes/1e6:.0f} MB')
print(f'Disk I/O time:        {total_io_ms:.0f} ms')
print(f'Generation time:      {t_total*1000:.0f} ms')
print(f'I/O fraction:         {total_io_ms/(t_total*1000):.1%}')
print(f'Unique experts seen:  {sum(len(c.accesses) for c in layer_caches.values())}')
print(f'Resident expert mem:  {total_resident/1e6:.1f} MB ({total_resident/1e9:.2f} GB)')
print(f'Tokens/sec:           {len(generated_tokens)/t_total:.2f}')

Per-layer cache statistics:
 layer  K  resident  accesses  hits  misses hit_rate  evictions  MB_loaded  io_ms  unique_experts  resident_MB
     0 62        57       253   196      57    77.5%          0      717.2  293.2              57        717.2
     1 62        62       257   195      62    75.9%          0      780.1  313.9              62        780.1
     2 62        60       255   195      60    76.5%          0      755.0  306.0              60        755.0
     3 62        57       254   197      57    77.6%          0      717.2  292.5              57        717.2
     4 62        56       252   196      56    77.8%          0      704.6  297.6              56        704.6
     5 62        53       250   197      53    78.8%          0      666.9  300.2              53        666.9
     6 62        53       250   197      53    78.8%          0      666.9  280.7              53        666.9
     7 62        56       253   197      56    77.9%          0      704.6  301.6   

## Step 8: Validation

Verify that this is a **true** K-expert cache — not a disguised full-bank-resident system.

Checks:
1. Generation succeeded (correct token count)
2. Every layer respects `resident_count ≤ K`
3. Cache statistics are populated (hits + misses > 0)
4. Active Metal memory is consistent with non-expert params + K cached experts,
   not the full ~12.3 GB expert bank
5. Actual resident expert memory matches expectation from K

In [10]:
metal_gb = mx.get_active_memory() / 1e9
max_resident = max(len(c.cache) for c in layer_caches.values())
total_resident_bytes = sum(c.resident_bytes() for c in layer_caches.values())
total_accesses_check = sum(c.hits + c.misses for c in layer_caches.values())

# Expected resident expert memory: K experts/layer × 16 layers × ~12.6 MB/expert
expected_resident_mb = CACHE_CAPACITY_K * 16 * (expert_bytes_total / 1e6)

# Full expert bank for reference
full_bank_mb = 64 * 16 * (expert_bytes_total / 1e6)

print(f'=== VALIDATION ===')
print(f'Cache capacity:           K = {CACHE_CAPACITY_K} per layer')
print(f'Max resident per layer:   {max_resident} experts')
print(f'Resident expert memory:   {total_resident_bytes/1e6:.1f} MB (actual from tensors)')
print(f'Expected resident memory: {expected_resident_mb:.1f} MB (K × 16 × per-expert)')
print(f'Full expert bank would be:{full_bank_mb:.0f} MB ({full_bank_mb/1e3:.1f} GB)')
print(f'Active Metal memory:      {metal_gb:.2f} GB')
print(f'Tokens generated:         {len(generated_tokens)}')
print(f'Total cache accesses:     {total_accesses_check}')
print()

# ── Assertions ──
errors = []

# 1. Generation succeeded
if len(generated_tokens) != MAX_NEW_TOKENS:
    errors.append(f'Expected {MAX_NEW_TOKENS} tokens, got {len(generated_tokens)}')
else:
    print(f'✓ Generated {len(generated_tokens)} tokens successfully')

# 2. Every layer respects K
for i, c in layer_caches.items():
    if len(c.cache) > CACHE_CAPACITY_K:
        errors.append(f'Layer {i}: {len(c.cache)} resident > K={CACHE_CAPACITY_K}')
if max_resident <= CACHE_CAPACITY_K:
    print(f'✓ All layers respect K={CACHE_CAPACITY_K} (max resident: {max_resident})')

# 3. Cache statistics are populated
if total_accesses_check == 0:
    errors.append('Cache has zero accesses — generation may not be using the cache')
else:
    print(f'✓ Cache statistics populated: {total_accesses_check} accesses')

# 4. No full expert bank residency
#    Non-expert params ~0.5 GB + cached experts + KV cache + overhead
#    Should be well under full expert bank (~12.3 GB)
max_expected_gb = (total_resident_bytes / 1e9) + 2.0  # generous: non-expert + KV + overhead
if metal_gb > full_bank_mb / 1000 * 0.8:  # >80% of full bank suggests full residency
    errors.append(
        f'Active Metal memory = {metal_gb:.2f} GB — '
        f'close to full expert bank ({full_bank_mb/1000:.1f} GB). '
        f'Expert bank may still be resident!'
    )
else:
    print(f'✓ Active Metal {metal_gb:.2f} GB << full bank {full_bank_mb/1000:.1f} GB — NOT full-bank-resident')

# 5. Actual resident memory is proportional to K
ratio = total_resident_bytes / (expected_resident_mb * 1e6) if expected_resident_mb > 0 else 0
if abs(ratio - 1.0) > 0.1:
    errors.append(
        f'Resident memory {total_resident_bytes/1e6:.1f} MB '
        f'!= expected {expected_resident_mb:.1f} MB (ratio: {ratio:.2f})'
    )
else:
    print(f'✓ Resident expert memory matches K (ratio: {ratio:.2f})')

if errors:
    print(f'\n✗ VALIDATION FAILED:')
    for e in errors:
        print(f'  - {e}')
    raise AssertionError('; '.join(errors))
else:
    print(f'\n✓ ALL VALIDATIONS PASSED — true K={CACHE_CAPACITY_K} per-layer cache confirmed')

=== VALIDATION ===
Cache capacity:           K = 62 per layer
Max resident per layer:   62 experts
Resident expert memory:   11626.6 MB (actual from tensors)
Expected resident memory: 12482.2 MB (K × 16 × per-expert)
Full expert bank would be:12885 MB (12.9 GB)
Active Metal memory:      12.58 GB
Tokens generated:         24
Total cache accesses:     4067

✓ Generated 24 tokens successfully
✓ All layers respect K=62 (max resident: 62)
✓ Cache statistics populated: 4067 accesses
✓ Resident expert memory matches K (ratio: 0.93)

✗ VALIDATION FAILED:
  - Active Metal memory = 12.58 GB — close to full expert bank (12.9 GB). Expert bank may still be resident!


AssertionError: Active Metal memory = 12.58 GB — close to full expert bank (12.9 GB). Expert bank may still be resident!